# Zero-LoRA Phase 2 (Colab, T4): reference pool, tests, pilot LoRA run
# Runtime > Change runtime type > T4 GPU. Run cells in order.

In [ ]:
!nvidia-smi | head -15
!git clone https://github.com/1forsithlord/zero-lora.git /content/zero-lora
%cd /content/zero-lora

In [ ]:
# Upload zero-lora-phase2-overlay.zip to /content, then copy it over the repo (new files only; Phase 1 files untouched)
!cd /content && unzip -q -o zero-lora-phase2-overlay.zip -d /content/zero-lora
!ls src/zerolora src/zerolora/losses

In [ ]:
# keep the Phase 1 pin: transformers 5.18.0 (newer versions break frozen_eval.py)
!pip -q install transformers==5.18.0 nltk huggingface_hub pyyaml scipy
!python tests/test_splits.py && python tests/test_refpool.py && python tests/test_lora_sanity.py

In [ ]:
# 2A. Build the reference pool (downloads CC3M shards + DTD/EuroSAT/Caltech101/Pets/CIFAR-100/Food-101 for the leakage check).
# Food-101 is ~5 GB; add --eval-datasets dtd eurosat caltech101 oxford_pets cifar100 --probe-max-images 5000 to shorten (limitation is then logged).
!python scripts/build_reference_pool.py --root /content/data --out data/reference_pool --shard-ids 0 1 2 --target-size 8000

In [ ]:
!cat data/reference_pool/pool_hash.json; head -c 1500 data/reference_pool/audit_log.jsonl

In [ ]:
# 2B pilot: plain LoRA on DTD, 20 steps, for timing/memory only
!PYTHONPATH=src python -m zerolora.train --root /content/data --task dtd --loss raw --lambda 0.0 --rank 4 --epochs 1 --max-steps 20 --lr 1e-4 --seed 0 --out results/pilot --tag pilot

In [ ]:
# 5-epoch plain LoRA (dev metrics only)
!PYTHONPATH=src python -m zerolora.train --root /content/data --task dtd --loss raw --lambda 0.0 --rank 4 --epochs 5 --lr 1e-4 --seed 0 --out results

In [ ]:
# Scale probe (NOT a tuned grid): one short run per loss to see the loss magnitudes. Selection of lambda happens later, on DEV only.
!PYTHONPATH=src python -m zerolora.train --root /content/data --task dtd --loss raw  --lambda 1e-3 --ref-pool data/reference_pool --epochs 1 --max-steps 30 --out results/scale_probe --tag probe
!PYTHONPATH=src python -m zerolora.train --root /content/data --task dtd --loss gnsp --lambda 1.0  --ref-pool data/reference_pool --epochs 1 --max-steps 30 --out results/scale_probe --tag probe
!grep -h l_pres results/scale_probe/*/metrics.jsonl | head

In [ ]:
# Save everything to Drive, then send results/*/summary.json and data/reference_pool/{pool_hash.json,audit_log.jsonl,manifest.json} to Claude
from google.colab import drive; drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/zero-lora-phase2 && cp -r results data/reference_pool/*.json* /content/drive/MyDrive/zero-lora-phase2/